# Evaluating LM Outputs using Rubric + LM Judges

Instead of using BLADE's `EntireAnalysisProcessed` code, we will try an evaluation implementation that reads straight from `multirun_analyses.json` and gives the results to an LLM judge.

## Setup

In [1]:
# imports
import json
import pandas as pd
import importlib.util
import sys
from os.path import join
from stat_genie.blade_pipeline.llms.config import llm
from stat_genie.blade_pipeline.additions.eval.extraction import \
    format_features, format_model_info
from stat_genie.blade_pipeline.additions.analysis.conclusion import \
    write_final_answer_code, make_conclusion

In [2]:
# define file paths
analysis_subdir_path = "analysis_output"
multirun_filename = "multirun_analyses.json"
# use multirun analyses file to get analysis code paths
multirun_analyses_path = join(analysis_subdir_path, multirun_filename)
with open(multirun_analyses_path, "r") as file:
    multirun_analyses = json.load(file)
num_analyses = multirun_analyses['n']
analysis_code_filenames = [f"llm_analysis_{i}.py" for i in range(num_analyses)]
analysis_code_paths = [join(analysis_subdir_path, filename) for filename in \
    analysis_code_filenames]
# get config details
llm_provider = "openai"
llm_model = "gpt-5-mini"
# create llm assistant
llm_assistant = llm(provider=llm_provider, model=llm_model)

[2025-11-13 10:29:51.24][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


## Extract Features **X** Used in Model

In [3]:
# create dict to store features
# features = {}

In [4]:
# # loop through analyses
# for i, analysis_code_path in enumerate(analysis_code_paths):
    
#     # create internal dict for analysis features
#     features[i] = {}
    
#     # get the features from each analysis
#     # this should include the independent and control variables
#     ind_vars = multirun_analyses['analyses'][str(i)]['cvars']['ivs']
#     control_vars = multirun_analyses['analyses'][str(i)]['cvars']['controls']

#     # get any lines from the transform code that represent transformations
#     # of the independent or control variables
#     transform_code = multirun_analyses['analyses'][str(i)]['transform_code']
#     # for each variable in ind_vars, check if it is transformed
#     # in the transform_code by using an LLM assistant
#     llm_assistant = llm(provider=llm_provider, model=llm_model)
#     for dict_idx, var in enumerate(ind_vars):
#         transform_responses = get_feature_transforms(llm_assistant,
#                                                      transform_code,
#                                                      var['columns'],
#                                                      var['description'])
#         ind_vars[dict_idx]['transform_code'] = [response.text[0].content for \
#             response in transform_responses]
        
#     # save updated independent variables in features dict
#     features[i]['independent_variables'] = ind_vars
    
#     # tkae same approach for control variables
#     for dict_idx, var in enumerate(control_vars):
#         transform_responses = get_feature_transforms(llm_assistant,
#                                                      transform_code,
#                                                      var['columns'],
#                                                      var['description'])
#         control_vars[dict_idx]['transform_code'] = [response.text[0].content for \
#             response in transform_responses]
    
#     # save updated control variables in features dict
#     features[i]['control_variables'] = control_vars

In [5]:
# view feature dictionary to ensure correctness
# features

## Extract Response *y* used in Model

In [6]:
# # loop through analyses
# for i, analysis_code_path in enumerate(analysis_code_paths):
    
#     # get the features from each analysis
#     # this should include the independent and control variables
#     response_vars = multirun_analyses['analyses'][str(i)]['cvars']['dv']

#     # get any lines from the transform code that represent transformations
#     # of the independent or control variables
#     transform_code = multirun_analyses['analyses'][str(i)]['transform_code']
#     # for each variable in response_vars, check if it is transformed
#     # in the transform_code by using an LLM assistant
#     llm_assistant = llm(provider=llm_provider, model=llm_model)
#     # for dict_idx, var in enumerate(response_vars):
#     transform_responses = get_feature_transforms(llm_assistant,
#                                                  transform_code,
#                                                  response_vars['columns'],
#                                                  response_vars['description'])
#     response_vars['transform_code'] = [response.text[0].content \
#         for response in transform_responses]

#     # save updated response variables in features dict
#     features[i]['response_variables'] = response_vars

In [7]:
# view feature dictionary to ensure correctness
# features

## Extract Features **X** and *y* Used in Model

In [8]:
features = format_features(multirun_analyses, num_analyses, llm_assistant)

In [9]:
features

{0: {'independent_variables': [{'description': 'Name femininity score (z-scored). Continuous index where higher values indicate a more feminine-sounding hurricane name. Primary independent variable testing whether more feminine names are associated with different fatality outcomes.',
    'columns': ['masfem_z'],
    'transform_code': ["# Create z-scored masfem variable to aid interpretation and reduce scale issues\nmasfem_mean = df['masfem'].mean()\nmasfem_std = df['masfem'].std(ddof=0)\nif masfem_std == 0 or np.isnan(masfem_std):\n    # fallback if no variance\n    df['masfem_z'] = df['masfem'] - masfem_mean\nelse:\n    df['masfem_z'] = (df['masfem'] - masfem_mean) / masfem_std"]}],
  'control_variables': [{'description': 'Binary indicator for whether the hurricane name is classified as female (1) or male (0). Included as a control to separate any discrete-name-gender effects from the continuous masfem measure.',
    'is_moderator': False,
    'moderator_on': None,
    'columns': ['ge

## Extract Model Class Used

In [10]:
model_info = format_model_info(multirun_analyses, num_analyses, llm_assistant)
model_info

{0: '{\n  "model_library": "statsmodels (statsmodels.formula.api as smf and statsmodels.api as sm)",\n  "model_class": "Primary: GLM NegativeBinomial (statsmodels.genmod.generalized_linear_model.GLM with family=NegativeBinomial()); Robustness checks: GLM Poisson (family=Poisson) with robust SEs and OLS on log(alldeaths+1) (statsmodels.regression.linear_model.OLS) with robust SEs.",\n  "model_parameters": "Formula: \'alldeaths ~ masfem_z + gender_female + wind + min + elapsedyrs + C(category) + C(source)\'. Families: NegativeBinomial() for primary model, Poisson() for robustness check. Robust SE type: cov_type=\'HC3\' used for Poisson and OLS fits. Default GLM link (log). Data preprocessing: dropna(subset=required), create log_deaths_plus1 = np.log(alldeaths + 1). C(category) and C(source) treat category and source as categorical. No additional hyperparameters specified.",\n  "model_formula_fitting_code": "formula = \'alldeaths ~ masfem_z + gender_female + wind + min + elapsedyrs + C(ca

## Extract Final Answer/Conclusion

Each of the BLADE tasks revolves around a question with the following format:

*What is the effect of [something] on [potential response]?*

It seems that often times the feature to use for the response is not deterministic; the model will have to use some sort of proxy to estimate it. The explanatory features are typically a little bit more clear, but still often require transformations and judgment calls on interpretation and use.

Importantly, this type of question ensures there is a binary answer. While the LLM data scientist does not explicitly spit out a yes/no value, it does write two functions: one which preprocesses the data and another that performs some sort of analysis. Theoretically, we could take the output of the analysis and inspect it to determine whether or not the feature of interest had an effect on the response.

In [11]:
# get path to the dataset
dataset_name = multirun_analyses['dataset_name']
dataset_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                    "datasets", dataset_name, "data.csv")

# load the dataset
data = pd.read_csv(dataset_path)

# create dictionaries to store the imported functions
transform_functions = {}
model_functions = {}

# loop through analyses
for i, analysis_code_path in enumerate(analysis_code_paths):
    # dynamically import the module
    spec = importlib.util.spec_from_file_location(f"llm_analysis_{i}",
                                                  analysis_code_path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[f"llm_analysis_{i}"] = module
    spec.loader.exec_module(module)
    
    # extract transform and model functions
    transform_functions[i] = module.transform
    model_functions[i] = module.model

In [12]:
# run the transform functions on the dataset
transformed_datasets = {}
for i, transform_func in transform_functions.items():
    transformed_datasets[i] = transform_func(data.copy()) # use copy of dataset

# run the model functions on the transformed datasets
model_results = {}
for i, model_func in model_functions.items():
    model_results[i] = model_func(transformed_datasets[i].copy()) # use copy

/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "
/accounts/grad/zachrewolinski/research/stat-genie/.venv/lib/python3.10/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


In [13]:
# view the first model result as a sanity check
model_results[0]

{'nb_model': <statsmodels.genmod.generalized_linear_model.GLMResultsWrapper at 0x75f49d15e050>,
 'poisson_robust': <statsmodels.genmod.generalized_linear_model.GLMResultsWrapper at 0x75f49d15d030>,
 'ols_log_outcome': <statsmodels.regression.linear_model.RegressionResultsWrapper at 0x75f49d15db70>,
 'model_dataframe':     ind  year      name    masfem  min  gender_mf  category  alldeaths  \
 0     0  1950      Easy  6.777778  958          1         3          2   
 1     1  1950      King  1.388889  955          0         3          4   
 2     2  1952      Able  3.833333  985          0         1          3   
 3     3  1953   Barbara  9.833333  987          1         1          1   
 4     4  1953  Florence  8.333333  985          1         1          0   
 ..  ...   ...       ...       ...  ...        ...       ...        ...   
 89   89  2008    Gustav  1.722222  951          0         2         52   
 90   90  2008       Ike  1.888889  935          0         2         84   
 91   

In [14]:
# create storage object for final answers
final_answer_code = {}

# read task from info.json in the dataset directory
info_json_path = join("..", "..", "src", "stat_genie", "blade_pipeline",
                      "datasets", dataset_name, "info.json")
with open(info_json_path, "r") as file:
    info_json = json.load(file)
task = info_json['research_questions']

for i in range(num_analyses):

    # get the independent and dependent variables from the dictionary made in
    # previous cells
    independent_variable = features[i]['independent_variables']
    dependent_variable = features[i]['response_variables']

    # get the model code
    model_code = multirun_analyses['analyses'][str(i)]['m_code']
    
    # get the model output from object made in previous cell
    model_output = model_results[i]
    
    # call the helper function
    final_answer_code[i] = write_final_answer_code(llm_assistant, task,
                                                   independent_variable,
                                                   dependent_variable,
                                                   model_code, model_output)

[2025-11-13 10:30:09.70][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-13 10:31:13.06][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  63.36 seconds
[2025-11-13 10:31:13.06][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-13 10:31:13.10][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-13 10:32:09.44][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  56.34 seconds
[2025-11-13 10:32:09.44][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)


In [15]:
# run the final answer code
final_answer_code

{0: 'def extract_final_answer(model_output):\n    """\n    Extracts the effect of \'masfem_z\' (name femininity, z-scored) on hurricane fatalities\n    from the provided model_output dict. Returns a dictionary with:\n      - "object": a dict of extracted statistics for the primary model (Negative Binomial if available,\n                  otherwise Poisson with robust SEs), plus robustness-check statistics from the Poisson\n                  and OLS(log(deaths+1)) models.\n      - "description": brief interpretation of the primary model result in context.\n    """\n    import numpy as np\n    import pandas as pd\n\n    def safe_conf_int(res, var):\n        """Return (low, high) for conf_int of var; handle different return types."""\n        try:\n            ci = res.conf_int()\n            # If conf_int returns DataFrame with index\n            if isinstance(ci, (pd.DataFrame, pd.Series)):\n                low, high = float(ci.loc[var, 0]), float(ci.loc[var, 1])\n            else:\n   

In [16]:
# loop through final answer code and dynamically execute the functions
final_answer_functions = {}
for i in range(num_analyses):
    # create a namespace dictionary to execute the code in
    namespace = {}
    
    # compile and execute the code
    compiled_code = compile(final_answer_code[i], f"<final_answer_code_{i}>", "exec")
    exec(compiled_code, namespace)
    
    # extract function from namespace
    final_answer_functions[i] = namespace['extract_final_answer']

# run the final answer functions on the model results
final_answers = [final_answer_functions[i](model_results[i]) for i in range(num_analyses)]

In [17]:
conclusions = {}
for i in range(num_analyses):

    # get the independent and dependent variables from the dictionary made in
    # previous cells
    independent_variable = features[i]['independent_variables']
    dependent_variable = features[i]['response_variables']

    # get the model code
    model_code = multirun_analyses['analyses'][str(i)]['m_code']
    
    # get the model interpretation code
    interpretation_code = final_answer_code[i]
    
    # get the interpretation output
    interpretation_output = final_answers[i]
    
    # call the helper function
    conclusions[i] = make_conclusion(llm_assistant, task, independent_variable,
                                     dependent_variable, model_code,
                                     interpretation_code, interpretation_output)

[2025-11-13 10:32:09.84][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)


[2025-11-13 10:32:20.92][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  11.08 seconds
[2025-11-13 10:32:20.92][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)
[2025-11-13 10:32:20.95][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-13 10:32:30.37][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  9.42 seconds
[2025-11-13 10:32:30.37][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)


In [18]:
conclusions

{0: '{\n  "answer": "No",\n  "justification": "The primary Negative Binomial estimate is small and negative (IRR ≈ 0.95) but far from statistically significant (p = 0.89) with a wide 95% CI (0.45–2.02) that includes 1. Poisson and OLS robustness checks give similar, non-significant results (n = 93). There is no evidence in these models that more feminine names lead to more fatalities (fewer precautions)."\n}',
 1: '{\n  "answer": "No",\n  "justification": "Neither the negative-binomial nor the OLS robustness model shows a statistically significant positive effect of name femininity on deaths (masfem_z p = 0.325 and p = 0.955 respectively); estimates are imprecise with CIs spanning null, so the analysis does not support the hypothesis."\n}'}

In [19]:
llm_judge = llm(provider=llm_provider, model=llm_model)

[2025-11-13 10:33:06.84][config_load.py:34 - blade_bench.llms.config_load:load_config][INFO] Loaded config from '/accounts/grad/zachrewolinski/research/stat-genie/config/llm_config.yml'.


In [21]:
data_head = data.head(10)

In [22]:
task

['Hurricanes with more feminine names are perceived as less threatening and hence lead to fewer precautionary measures by the general public.']

In [29]:
judge_system_prompt = (
    "You are a meticulous research design evaluator. "
    "Your role is to compare two experimental trials methodologically **and interpretively**.\n\n"
    "You will go through the following reasoning plan step-by-step (internally):\n"
    "1. Understand the research question and dataset context.\n"
    "2. Examine independent, control, and response variables for both trials.\n"
    "3. Analyze the model specifications for structural or methodological similarity.\n"
    "4. Focus more on the content, less on the format.\n"
    "5. Assess whether the trials' conclusions are logically consistent given their setups.\n"
    "6. Detect whether either input is None, invalid, erroneous, or incomplete.\n"
    "   - If **one trial** shows errors or missing components but the other is valid, "
    "     impose a **strong penalty** (reduce all category scores by at least 1 point, "
    "     and cap overall similarity at 2).\n"
    "7. Synthesize your evaluation across all components.\n"
    "8. Output a numerical rating for each category.\n\n"
    "DO NOT include your reasoning — only the final JSON object.\n\n"
    "Scoring scale:\n"
    "1 = completely different\n"
    "2 = somewhat different\n"
    "3 = moderately similar\n"
    "4 = very similar\n"
    "5 = almost identical\n\n"
    "Return output **strictly in JSON format**:\n"
    "{\n"
    "  \"independent_variables\": <number>,\n"
    "  \"control_variables\": <number>,\n"
    "  \"response_variables\": <number>,\n"
    "  \"model_specification\": <number>,\n"
    "  \"conclusions\": <number>,\n"
    "  \"overall_similarity\": <number>\n"
    "}"
)

In [30]:
judge_user_prompt = (
    f"Research Question / Context:\n{task}\n\n"
    "Here is a sample of the dataset to understand the structure and variables:\n"
    f"{data_head}\n\n"
    "Compare the two trials methodologically and interpretively based on the provided variables, model specifications, and conclusions.\n\n"
    "==================== TRIAL 0 ====================\n\n"
    "Independent Variables:\n"
    f"{features[0]['independent_variables']}\n\n"
    "Control Variables:\n"
    f"{features[0]['control_variables']}\n\n"
    "Response Variables:\n"
    f"{features[0]['response_variables']}\n\n"
    "Model Specification:\n"
    f"{model_info[0]}\n\n"
    "Conclusion:\n"
    f"{conclusions[0]}\n\n"
    "==================== TRIAL 1 ====================\n\n"
    "Independent Variables:\n"
    f"{features[1]['independent_variables']}\n\n"
    "Control Variables:\n"
    f"{features[1]['control_variables']}\n\n"
    "Response Variables:\n"
    f"{features[1]['response_variables']}\n\n"
    "Model Specification:\n"
    f"{model_info[1]}\n\n"
    "Conclusion:\n"
    f"{conclusions[1]}\n\n"
    "Now, following your reasoning plan, provide similarity ratings as JSON only."
)

In [31]:
final_scores = llm_judge.generate([{"role": "system",
                                        "content": judge_system_prompt},
                                       {"role": "user",
                                        "content": judge_user_prompt}])

[2025-11-13 10:40:55.27][base.py:60 - stat_genie.blade_pipeline.llms.base:generate][INFO] Sending openai API Request (gpt-5-mini)
[2025-11-13 10:41:07.41][base.py:67 - stat_genie.blade_pipeline.llms.base:generate][DEBUG] openai response took  12.13 seconds
[2025-11-13 10:41:07.41][base.py:71 - stat_genie.blade_pipeline.llms.base:generate][API] Called openai API (gpt-5-mini)


In [33]:
final_scores.text[0].content

'{\n  "independent_variables": 5,\n  "control_variables": 4,\n  "response_variables": 5,\n  "model_specification": 4,\n  "conclusions": 5,\n  "overall_similarity": 5\n}'